# NZCCDv2 merge (maintainer only)

**Starting template - not finished.** Check the results before publishing anything.

`new_DSAS.ipynb` writes one set of outputs per person per area, into `DataUpdatev2/<name>/`, tagged with the selection that produced it:

- `NZCCDv2_<tag>.shp`
- `ratesv2_<tag>.shp`
- `intersectsv2_<tag>.shp`

Git cannot merge shapefiles, so those files are never combined by branching. This notebook does it instead: it collects everyone's tagged outputs from `DataUpdatev2` and its subfolders, stitches them back together with geopandas, and writes a single national dataset.

Only the project maintainer runs this. Students hand over their `DataUpdatev2` folder; they do not merge into the shared dataset themselves.

Outputs land in `OUT_DIR` as `NZCCDv2.shp`, `ratesv2.shp` and `intersectsv2.shp`.

In [ ]:
# 1) Settings
from pathlib import Path

import geopandas as gpd
import pandas as pd
import shapely

# Folders to collect per-area outputs from. Add one entry per person or per run.
# These can be local folders or paths on the shared drive.
INPUT_DIRS = [
    Path("DataUpdatev2"),
]

# Where the combined dataset is written.
OUT_DIR = Path("DataUpdatev2/merged")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The national dataset the per-area runs were cut from.
V1_PATH = Path("Data for testing/NZCCDv1.shp")

SHORELINE_GLOB = "NZCCDv2_*.shp"
RATES_GLOB = "ratesv2_*.shp"
POINTS_GLOB = "intersectsv2_*.shp"


def _norm(text):
    return "".join(ch for ch in str(text).lower() if ch.isalnum())


def geom_hash(geom):
    if geom is None:
        return None
    try:
        return shapely.to_wkb(geom, hex=True)
    except Exception:
        return None


def find_inputs(pattern):
    found = []
    for folder in INPUT_DIRS:
        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue
        found.extend(sorted(folder.glob(pattern)))
    return found


def tag_from_path(path, prefix):
    return path.stem[len(prefix):] if path.stem.startswith(prefix) else path.stem

In [ ]:
# 2) Find the per-area outputs
shoreline_files = find_inputs(SHORELINE_GLOB)
rates_files = find_inputs(RATES_GLOB)
points_files = find_inputs(POINTS_GLOB)

for label, files in [("shorelines", shoreline_files), ("rates", rates_files), ("intersects", points_files)]:
    print(f"{label}: {len(files)} file(s)")
    for path in files:
        print(f"  - {path}")

if not shoreline_files:
    raise FileNotFoundError("No NZCCDv2_*.shp files found. Check INPUT_DIRS.")

In [ ]:
# 3) Merge the shorelines
# Start from NZCCDv1 so the result stays national, then add each per-area run on top.
base = gpd.read_file(V1_PATH)
if "Location" not in base.columns and "Site" in base.columns:
    base["Location"] = base["Site"]
base["SourceTag"] = "NZCCDv1"
print(f"NZCCDv1: {len(base):,} rows")

frames = [base]
for path in shoreline_files:
    g = gpd.read_file(path)
    if g.crs is not None and base.crs is not None and str(g.crs) != str(base.crs):
        g = g.to_crs(base.crs)
    g["SourceTag"] = tag_from_path(path, "NZCCDv2_")
    frames.append(g)
    print(f"{path.name}: {len(g):,} rows")

combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=base.crs)

for col in ["Region", "Location", "Date", "GeomHash"]:
    if col not in combined.columns:
        combined[col] = pd.NA
combined["GeomHash"] = combined["GeomHash"].where(combined["GeomHash"].notna(), combined.geometry.apply(geom_hash))

# A shoreline is the same feature if it is the same place, same date and same geometry.
dedupe_key = pd.DataFrame({
    "Region": combined["Region"].astype(str).map(_norm),
    "Location": combined["Location"].astype(str).map(_norm),
    "Date": pd.to_datetime(combined["Date"], errors="coerce").astype(str),
    "GeomHash": combined["GeomHash"].astype(str),
})
duplicated = dedupe_key.duplicated(keep="first")
print(f"Dropping {int(duplicated.sum()):,} duplicate shoreline rows")

shorelines_merged = combined[~duplicated].reset_index(drop=True)
print(f"Merged shorelines: {len(shorelines_merged):,} rows")
shorelines_merged["SourceTag"].value_counts()

In [ ]:
# 4) Merge the rates and intersect points
# Different areas produce different transects, so IDs should not repeat. If they do,
# two runs covered the same transect and the overlap needs a decision - see the warning.
def merge_tables(files, id_col, prefix, label):
    if not files:
        print(f"No {label} files found - skipping")
        return None

    frames = []
    crs = None
    for path in files:
        g = gpd.read_file(path)
        if crs is None:
            crs = g.crs
        elif g.crs is not None and str(g.crs) != str(crs):
            g = g.to_crs(crs)
        g["SourceTag"] = tag_from_path(path, prefix)
        frames.append(g)
        print(f"{path.name}: {len(g):,} rows")

    out = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=crs)

    if id_col not in out.columns:
        print(f"WARNING: {label} has no {id_col} column, nothing de-duplicated")
    else:
        duplicated = out.duplicated(subset=[id_col], keep="first")
        if duplicated.any():
            # TODO: agree a rule for overlapping runs (newest wins? most shorelines wins?).
            print(f"WARNING: {int(duplicated.sum()):,} repeated {id_col} values in {label}; keeping the first")
            print(out.loc[duplicated, [id_col, "SourceTag"]].head(10))
        out = out[~duplicated].reset_index(drop=True)

    print(f"Merged {label}: {len(out):,} rows")
    return out


rates_merged = merge_tables(rates_files, "UniqueID", "ratesv2_", "rates")
points_merged = merge_tables(points_files, "Unique_ID", "intersectsv2_", "intersects")

In [ ]:
# 5) Save the combined dataset
shorelines_out = OUT_DIR / "NZCCDv2.shp"
shorelines_merged.to_file(shorelines_out)
print(f"Saved {shorelines_out}")

if rates_merged is not None:
    rates_merged.to_file(OUT_DIR / "ratesv2.shp")
    rates_merged.drop(columns="geometry").to_csv(OUT_DIR / "ratesv2.csv", index=False)
    print(f"Saved {OUT_DIR / 'ratesv2.shp'}")

if points_merged is not None:
    points_merged.to_file(OUT_DIR / "intersectsv2.shp")
    points_merged.drop(columns="geometry").to_csv(OUT_DIR / "intersectsv2.csv", index=False)
    print(f"Saved {OUT_DIR / 'intersectsv2.shp'}")